In [0]:
spark.conf.set("fs.azure.account.key.ecommercedeproj1998.dfs.core.windoes.net","")

In [0]:
df_silver = spark.read.format("delta").load(
    "abfss://silver@ecommercedeproj1998.dfs.core.windows.net/online_retail_cleaned"
)
df_silver.printSchema()
df_silver.show(5)

In [0]:
dim_customer = df_silver.select("CustomerID", "Country") \
    .dropDuplicates(["CustomerID"])

dim_customer.write.format("delta").mode("overwrite").save(
    "abfss://gold@ecommercedeproj1998.dfs.core.windows.net/dim_customer"
)
print("dim_customer written:", dim_customer.count(), "rows")

In [0]:
dim_product = df_silver.select("StockCode", "Description") \
    .dropDuplicates(["StockCode"])

dim_product.write.format("delta").mode("overwrite").save(
    "abfss://gold@ecommercedeproj1998.dfs.core.windows.net/dim_product"
)
print("dim_product written:", dim_product.count(), "rows")

In [0]:
from pyspark.sql.functions import year, month, dayofmonth, quarter, dayofweek

dim_date = df_silver.select("InvoiceDate").dropDuplicates()

dim_date = dim_date.withColumn("Year", year("InvoiceDate")) \
    .withColumn("Month", month("InvoiceDate")) \
    .withColumn("Day", dayofmonth("InvoiceDate")) \
    .withColumn("Quarter", quarter("InvoiceDate")) \
    .withColumn("DayOfWeek", dayofweek("InvoiceDate"))

In [0]:
from pyspark.sql.functions import col

fact_sales = df_silver.select(
    "InvoiceNo", "StockCode", "CustomerID", "InvoiceDate", 
    "Quantity", "UnitPrice", "TransactionType"
).withColumn("TotalAmount", col("Quantity") * col("UnitPrice"))

fact_sales.write.format("delta").mode("overwrite").save(
    "abfss://gold@ecommercedeproj1998.dfs.core.windows.net/fact_sales"
)
print("fact_sales written:", fact_sales.count(), "rows")

In [0]:
fact_sales.show(5)